In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
 
INPUT_CSV  = './output/dataset.csv'
OUTPUT_CSV = './output/dataset_final.csv'
 
VICON_ANGLE_JOINTS = [
    'Head','Left_head','Right_head','Left_neck','Right_neck',
    'Left_clavicle','Right_clavicle','Thorax','Left_thorax','Right_thorax',
    'Pelvis','Left_pelvis','Right_pelvis','Left_hip','Right_hip',
    'Left_femur','Right_femur','Left_knee','Right_knee',
    'Left_tibia','Right_tibia','Left_ankle','Right_ankle',
    'Left_foot','Right_foot','Left_toe','Right_toe',
    'Left_shoulder','Right_shoulder','Left_elbow','Right_elbow',
    'Left_radius','Right_radius','Left_wrist','Right_wrist',
    'Left_upperhand','Right_upperhand','Left_hand','Right_hand',
]
ANGLE_COLS = [f'{j}_{ax}' for j in VICON_ANGLE_JOINTS for ax in ['X', 'Y', 'Z']]
 
df = pd.read_csv(INPUT_CSV)

In [2]:
df_corretos = df[df['label'] == 0].copy()
df_incorretos = df[df['label'] == 1].copy()

def gerar_mock_data(df_base, quantidade_desejada):
    execucoes = [group for _, group in df_base.groupby(['movement', 'subject', 'execution'])]
    novas_execucoes = []
    for i in range(quantidade_desejada):
        base = execucoes[np.random.randint(0, len(execucoes))].copy()
        ruido = np.random.normal(0, 0.01, size=(len(base), len(ANGLE_COLS)))
        base[ANGLE_COLS] = base[ANGLE_COLS] + ruido
        base['source_subject'] = base['subject']   # origem real preservada
        base['subject']        = base['subject'] + 2000 + i
        base['execution']      = base['execution'] + 2000 + i
        novas_execucoes.append(base)
    return pd.concat(novas_execucoes)

# Dados reais recebem source_subject = -1 (sem origem mock)
df['source_subject'] = -1

print("Gerando novos dados..")
mock_corretos   = gerar_mock_data(df_corretos, 1500)
mock_incorretos = gerar_mock_data(df_incorretos, 1500)

# Junta o original com o mock
df_final = pd.concat([df, mock_corretos, mock_incorretos]).reset_index(drop=True)
df = df_final

CAMINHO_FINAL = './output/dataset_final_5000.csv'
df.to_csv(CAMINHO_FINAL, index=False)

print(f"success")
print(f"O arquivo foi salvo em: {CAMINHO_FINAL}")
print(f"Tamanho final do arquivo: {df.shape[0]} linhas (frames).")
print(f"Colunas: {df.columns.tolist()}")



/tmp/ipykernel_128703/1960687653.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['source_subject'] = -1
/tmp/ipykernel_128703/1960687653.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  base['source_subject'] = base['subject']   # origem real preservada
/tmp/ipykernel_128703/1960687653.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de

Gerando novos dados..


/tmp/ipykernel_128703/1960687653.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  base['source_subject'] = base['subject']   # origem real preservada
/tmp/ipykernel_128703/1960687653.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  base['source_subject'] = base['subject']   # origem real preservada
/tmp/ipykernel_128703/1960687653.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once us

success
O arquivo foi salvo em: ./output/dataset_final_5000.csv
Tamanho final do arquivo: 1213454 linhas (frames).
Colunas: ['movement', 'subject', 'execution', 'label', 'frame', 'Head_X', 'Head_Y', 'Head_Z', 'Left_head_X', 'Left_head_Y', 'Left_head_Z', 'Right_head_X', 'Right_head_Y', 'Right_head_Z', 'Left_neck_X', 'Left_neck_Y', 'Left_neck_Z', 'Right_neck_X', 'Right_neck_Y', 'Right_neck_Z', 'Left_clavicle_X', 'Left_clavicle_Y', 'Left_clavicle_Z', 'Right_clavicle_X', 'Right_clavicle_Y', 'Right_clavicle_Z', 'Thorax_X', 'Thorax_Y', 'Thorax_Z', 'Left_thorax_X', 'Left_thorax_Y', 'Left_thorax_Z', 'Right_thorax_X', 'Right_thorax_Y', 'Right_thorax_Z', 'Pelvis_X', 'Pelvis_Y', 'Pelvis_Z', 'Left_pelvis_X', 'Left_pelvis_Y', 'Left_pelvis_Z', 'Right_pelvis_X', 'Right_pelvis_Y', 'Right_pelvis_Z', 'Left_hip_X', 'Left_hip_Y', 'Left_hip_Z', 'Right_hip_X', 'Right_hip_Y', 'Right_hip_Z', 'Left_femur_X', 'Left_femur_Y', 'Left_femur_Z', 'Right_femur_X', 'Right_femur_Y', 'Right_femur_Z', 'Left_knee_X', 'Left

In [3]:
print(df['subject'].unique())

[   1    2    3 ... 3495 3496 3508]


In [4]:
# Análise dos metadados do novo dataset
print("=== ANÁLISE DO DATASET FINAL ===\n")

# Contagem de registros por label
print("Distribuição por label (0=correto, 1=incorreto):")
print(df['label'].value_counts().sort_index())
print(f"Total: {len(df)} frames\n")

# Movimentos únicos
print(f"Movimentos únicos: {sorted(df['movement'].unique())}")
print(f"Total de movimentos: {df['movement'].nunique()}\n")

# Sujeitos únicos
print(f"Sujeitos únicos: {df['subject'].nunique()}")
print(f"Range: {df['subject'].min()} - {df['subject'].max()}\n")

# Execuções por movimento
print("Média de execuções por movimento:")
exec_por_mov = df.groupby('movement')['execution'].nunique()
print(exec_por_mov)
print(f"Média: {exec_por_mov.mean():.2f}\n")

# Breakdown completo
print("Breakdown por movement e label:")
breakdown = df.groupby(['movement', 'label']).size().unstack(fill_value=0)
breakdown['total'] = breakdown.sum(axis=1)
print(breakdown)

=== ANÁLISE DO DATASET FINAL ===

Distribuição por label (0=correto, 1=incorreto):
label
0    606283
1    607171
Name: count, dtype: int64
Total: 1213454 frames

Movimentos únicos: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]
Total de movimentos: 10

Sujeitos únicos: 1327
Range: 1 - 3508

Média de execuções por movimento:
movement
1     268
2     286
3     250
4     292
5     287
6     299
7     273
8     314
9     294
10    280
Name: execution, dtype: int64
Média: 284.30

Breakdown por movement e label:
label         0      1   total
movement                      
1         67351  65344  132695
2         56545  57661  114206
3         55056  59446  114502
4         65348  71855  137203
5         73658  81765  155423
6         56443  50255  106698
7         65761  52580  118341
8         56769  57791  114560
9         60016  61152  121168
10        49336  49322   98658


In [5]:
real_df = df[df['subject'] < 1000]
mock_df = df[df['subject'] >= 1000]
 
print("=== COMPARAÇÃO DE MOVIMENTOS: REAL vs MOCK ===\n")
 
print("Breakdown REAL (subject < 1000):")
breakdown_real = real_df.groupby(['movement', 'label']).size().unstack(fill_value=0)
breakdown_real['total'] = breakdown_real.sum(axis=1)
print(breakdown_real)
print(f"Total frames reais: {len(real_df)}\n")
 
print("Breakdown MOCK (subject >= 1000):")
breakdown_mock = mock_df.groupby(['movement', 'label']).size().unstack(fill_value=0)
breakdown_mock['total'] = breakdown_mock.sum(axis=1)
print(breakdown_mock)
print(f"Total frames mock: {len(mock_df)}\n")
 
exec_real = real_df.groupby('movement')['execution'].nunique()
exec_mock = mock_df.groupby('movement')['execution'].nunique()
 
print("Execuções por movimento:")
print("REAL:")
print(exec_real)
print("\nMOCK:")
print(exec_mock)
print("\nDiferença (Mock - Real):")
print(exec_mock - exec_real)


=== COMPARAÇÃO DE MOVIMENTOS: REAL vs MOCK ===

Breakdown REAL (subject < 1000):
label         0      1  total
movement                     
1         27063  28116  55179
2         23073  22819  45892
3         23717  23255  46972
4         27884  27014  54898
5         28928  33749  62677
6         21146  19842  40988
7         24917  23275  48192
8         21802  20558  42360
9         23286  24426  47712
10        20719  19857  40576
Total frames reais: 485446

Breakdown MOCK (subject >= 1000):
label         0      1  total
movement                     
1         40288  37228  77516
2         33472  34842  68314
3         31339  36191  67530
4         37464  44841  82305
5         44730  48016  92746
6         35297  30413  65710
7         40844  29305  70149
8         34967  37233  72200
9         36730  36726  73456
10        28617  29465  58082
Total frames mock: 728008

Execuções por movimento:
REAL:
movement
1     10
2     10
3     10
4     10
5     10
6     10
7     10
8     1